# Mesh

Reusable function definitions only. Original project inputs, run settings and saved outputs are not included. Running these cells defines functions; it does not read a dataset. Start with the [synthetic walkthrough](../examples/synthetic_walkthrough.ipynb) for a complete small run.


In [ ]:
from pathlib import Path
import sys

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "helper" / "pipeline_utils.py").is_file()
             and (p / "examples" / "synthetic_demo.py").is_file()), None)
if ROOT is None:
    raise RuntimeError("Start the notebook inside this repository.")
for folder in (ROOT, ROOT / "Invert-PropOnly-saveCSV", ROOT / "Invert-Prop+cond-saveCSV"):
    if str(folder) not in sys.path:
        sys.path.insert(0, str(folder))


In [ ]:
import numpy as np
import torch

from Helper_invert_1d import initialize_inversion, fitness_s, unpack_qpso_solutions


def invert_first_output_mesh_select_Q(
    x_scaler_path: str,
    y_scaler_path: str,
    model_path: str,
    x_fixed_raw: np.ndarray,        # raw fixed inputs (len 10)
    actual_x_adj_raw: np.ndarray,   # raw adjustable inputs (len 4)
    target_raw: float,              # desired raw value of output0
    mesh_dims: tuple[int, int, int, int],
    Q_bounds: tuple[float, float],  # (Q_min, Q_max) in raw space
    k: int,                         # number of top solutions to return
    q: int,                         # oversampling factor (select k*q cells)
    MAX_ITER: int,                  # Newton max iterations per cell
    STEP_SIZE: float,               # Newton step size multiplier
    TOL: float,                     # Newton convergence tolerance
    hidden_dims: list[int],
    activation: str,
    x_adj_intervals_raw: np.ndarray # raw [min,max] intervals for x0,x1,x2,x3
) -> list[tuple[np.ndarray, np.ndarray, float]]:
    """
    1) Initialize scalers, model, scaling, “actual” prediction & residual,
       and scaled search bounds via initialize_inversion.
    2) Build a regular 4D mesh over [x0_s, x2_s, x3_s, Q].
    3) Evaluate fitness_s at each cell’s center, pick top k*q.
    4) Refine each selected center via Newton–Raphson (clipped to cell).
    5) Use unpack_qpso_solutions to map the refined scaled candidates
       back to raw inputs, full outputs, and residuals.
    6) Return a list whose first element is the actual test‐data pair,
       followed by the top‐k refined solutions (sorted by raw residual).
    """
    # -- [1] Load everything & get initial data, model, and bounds ----------
    init = initialize_inversion(
        x_scaler_path, y_scaler_path, model_path,
        hidden_dims, activation,
        x_fixed_raw, actual_x_adj_raw,
        target_raw, x_adj_intervals_raw, Q_bounds
    )
    model         = init["model"]
    x_fixed_s     = init["x_fixed_s"]
    means_x       = init["means_x"]
    scales_x      = init["scales_x"]
    y_scaler      = init["y_scaler"]
    results       = init["results"]       # starts with (actual_x_adj_raw, preds_act_raw, resid_act)
    min_vec       = init["min_vec"]       # [x0_s_min, x2_s_min, x3_s_min, Q_min]
    max_vec       = init["max_vec"]       # [x0_s_max, x2_s_max, x3_s_max, Q_max]
    target_scaled = init["target_scaled"] # (target_raw - y_mean)/y_scale

    # -- [2] Build 4D mesh of cell centers & bounds -------------------------
    mesh = np.array(mesh_dims, dtype=int)
    steps = (max_vec - min_vec) / mesh
    centers_cells = []
    # iterate over all cells
    for idx in np.ndindex(*mesh):
        idx = np.array(idx)
        cmin   = min_vec + idx * steps
        cmax   = cmin + steps
        center = cmin + 0.5 * steps
        centers_cells.append((center, cmin, cmax))

    # -- [3] Score centers & select top k*q by fitness ----------------------
    scored = [
        (fitness_s(center, model, x_fixed_s, means_x, scales_x, target_scaled),
         center, cmin, cmax)
        for center, cmin, cmax in centers_cells
    ]
    scored.sort(key=lambda t: t[0])
    selected = scored[: k * q]  # pick the best k*q cells

    # -- [4] Refine each selected center via Newton–Raphson -----------------
    refined_centers = []
    for _, center, cmin, cmax in selected:
        # promote to torch tensor for autograd
        x_var = torch.tensor(center, requires_grad=True, dtype=torch.float32)
        low   = torch.from_numpy(cmin).float()
        high  = torch.from_numpy(cmax).float()

        for _ in range(MAX_ITER):
            if x_var.grad is not None:
                x_var.grad.zero_()

            # unpack variables
            x0_s, x2_s, x3_s, Q = x_var
            # reconstruct x1_s from Q and raw x0
            x0_raw = x0_s * scales_x[0] + means_x[0]
            x1_raw = Q     * x0_raw
            x1_s   = (x1_raw - means_x[1]) / scales_x[1]

            # build full network input
            inp = torch.cat([
                torch.stack([x0_s, x1_s, x2_s, x3_s]),
                torch.from_numpy(x_fixed_s).float()
            ])

            # compute scaled output0 error
            f1 = model(inp)[0]
            g  = f1 - target_scaled
            if abs(g.item()) < TOL:
                break  # converged

            # Newton step: Δ = (g / ∥∇g∥²) ∇g • step_size
            g.backward()
            grad  = x_var.grad
            denom = grad.dot(grad)
            if denom.abs().item() < 1e-12:
                break  # gradient too small

            delta = (g / denom) * grad * STEP_SIZE
            x_var = (x_var - delta).clamp(low, high).detach().requires_grad_(True)

        # collect refined center
        refined_centers.append(x_var.detach().cpu().numpy())

    # -- [5] Map refined scaled candidates → raw results + sort & trim ------
    others = unpack_qpso_solutions(
        refined_centers,
        means_x,
        scales_x,
        x_fixed_s,
        model,
        y_scaler,
        target_raw
    )
    # sort by raw residual, then take top-k
    others_sorted = sorted(others, key=lambda t: t[2])[:k]

    # return actual followed by top-k refined solutions
    return [results[0]] + others_sorted
